In [1]:
import os
from tqdm.notebook import tqdm

import torch
import numpy as np
import plotly.express as px
from torch import nn, Tensor
import matplotlib.colors as mcolors
from matplotlib import pyplot as plt
from torchvision.transforms import v2

from src import configs as cfg
from src import dataset, models

/root/repos/raidium_challenge/.venv/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
def load_chkpt(chkpt_pth: str) -> torch.nn.Module:
    chkpt = torch.load(chkpt_pth, weights_only=False)
    model_cfg = cfg.ModelConfig(**chkpt["model_cfg"])
    model = models.mk_model_from_cfg(model_cfg)
    model.load_state_dict(chkpt["model"])
    model = model.eval()
    return model

In [3]:
train_cfg = cfg.TrainingConfig(batch_size=2)
loaders = dataset.mk_segmentation_data_loaders(train_cfg)
x, y_true = next(iter(loaders["train"]))
batch = dataset.preprocess_batch({"x": x, "y_true": y_true})

CHKPT_DIRECTORY = "checkpoints/unet/vague-feather-528/"
N_CHKPT_TO_PLT = -1
chkpt_filenames = os.listdir(CHKPT_DIRECTORY)[:N_CHKPT_TO_PLT]
segs_buffer = torch.empty(
    len(chkpt_filenames), train_cfg.batch_size, 256, 256,
    dtype=torch.uint8,
    device=cfg.DEVICE,
)

with torch.no_grad():
    for chkpt_idx, chkpt_filename in tqdm(enumerate(chkpt_filenames)):
        chkpt_pth = os.path.join(CHKPT_DIRECTORY, chkpt_filename)
        model = load_chkpt(chkpt_pth)
        segs_buffer[chkpt_idx] = (
            model(batch)["y_pred"]
            .argmax(dim=1)
            .to(dtype=torch.uint8)
        )

sampling method: shuffle
train_dl_kwargs: {'shuffle': True, 'batch_size': 2}


0it [00:00, ?it/s]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import plotly.express as px

N_SAMPLES_TO_SHOW = 10
COLOR_MAP_NAME = "rainbow"


# For segmentation labels (categorical)
seg_norm = Normalize(
    vmin=0,
    vmax=cfg.N_CLASSES - 1,   # important: categorical, fixed range
)

segs_buffer_np = segs_buffer.cpu().numpy()  # [B, H, W]

colored_seg_buffer = plt.cm.get_cmap(COLOR_MAP_NAME)(
    seg_norm(segs_buffer_np)
)  # -> [B, H, W, 4]

# For grayscale images
# Adjust depending on your preprocessing
x_np = batch["x"].cpu().numpy()  # [B, H, W, C] or [B, H, W]
x_norm = Normalize(
    vmin=x_np.max(),
    vmax=x_np.min(),   # or [-3, 3] if normalized, or [0, 255] if uint8
)

colored_x = plt.cm.gray(x_norm(x_np))  # -> [B, H, W, 4]

seg_mask = (segs_buffer_np == 0)[..., None]  # background mask

test_img_buffer = np.where(
    seg_mask,
    colored_x,
    colored_seg_buffer,
)

# Remove alpha channel
test_img_buffer = test_img_buffer[..., :3]  # [B, H, W, 3]

print(test_img_buffer.shape)

fig = px.imshow(
    test_img_buffer,
    animation_frame=0,
    facet_col=1,
    facet_col_wrap=train_cfg.batch_size,
)

fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 30
fig


/tmp/ipykernel_7434/2084829379.py:18: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.



ValueError: minvalue must be less than or equal to maxvalue